# HIPPIE tutorial: train on your own data

The companion notebook (`cross_dataset_tutorial.ipynb`) uses the **pretrained** checkpoint.
This one trains a HIPPIE model **from scratch** on a single dataset in the canonical CSV
layout, then embeds and evaluates it. It mirrors `scripts/train.py` step by step, so anything
here also works from the command line:

```bash
python scripts/train.py --dataset hull_cell_type --data-dir ./datasets_hippie \
    --output checkpoints/hull.ckpt --epochs 100
```


## 1. Setup

```bash
pip install -e ".[viz]"
```

In [ ]:
import os, re
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from hippie.dataloading import MultiModalEphysDataset, none_safe_collate
from hippie.augmentations import AugmentedMultiModalEphysDataset
from hippie.multimodal_model import (
    CVAEConfig, ExperimentConfigs, MultiModalCVAE, MultiModalCVAETrainModule)
from hippie.inference import TECHNOLOGY_IDS, HIPPIEClassifier, select_k_via_cv

## 2. Configuration

The defaults are the paper's locked production hyperparameters (`config`, `z_dim=30`,
`beta=1.0`). `tech` is the recording technology of your dataset; `num_sources=4` matches the
released checkpoint's source vocabulary so the technology covariate stays compatible.

In [ ]:
DATASET     = 'hull_cell_type'   # a folder under DATA_DIR with the 4 canonical CSVs
# Locate datasets_hippie regardless of the kernel's working directory.
# VS Code often runs a notebook from the workspace root, not the repo root,
# so we search upward and also look inside a HIPPIE/ subfolder.
def _find_dir(name='datasets_hippie', max_up=6):
    here = os.path.abspath(os.getcwd())
    for k in range(max_up + 1):
        base = os.path.abspath(os.path.join(here, *(['..'] * k)))
        for cand in (os.path.join(base, name), os.path.join(base, 'HIPPIE', name)):
            if os.path.isdir(cand):
                return cand
    raise FileNotFoundError(
        f"Could not find '{name}'. Open the HIPPIE folder as your VS Code "
        f"workspace, or set this path to an absolute location.")
DATA_DIR = _find_dir()
TECH        = 'neuropixels'      # one of TECHNOLOGY_IDS
OUTPUT      = 'checkpoints/my_hull_run.ckpt'

CONFIG_NAME = 'class_decoder_source_bn_aug_reg'   # production default
Z_DIM       = 30
BETA        = 1.0
NUM_SOURCES = 4
EPOCHS      = 30      # demo value (~minutes on CPU); production uses 100, ideally on GPU
BATCH_SIZE  = 32
SEED        = 42

pl.seed_everything(SEED, workers=True)
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'

## 3. Load and clean the data

Same `{waveforms,isi_dist,acg,labels}.csv` layout as the inference notebook. We strip the
byte-string label wrapper (`b'PkC_ss'` -> `PkC_ss`), drop unlabeled rows, and integer-encode
the labels with `LabelEncoder`. Modalities are passed as **raw** arrays: the dataset class
resamples and normalizes them internally (`log(x+1)` for ISI, min-max to `[-1, 1]`).

In [ ]:
_BYTESTR = re.compile(r"""^b(['\"])(.*)\1$""")

def clean_labels(raw):
    cleaned = np.array([
        (_BYTESTR.match(s).group(2) if _BYTESTR.match(s) else s) for s in raw.astype(str)])
    keep = ~np.isin(cleaned, ['', 'unlabeled', 'nan', 'None'])
    return cleaned, keep

d = os.path.join(DATA_DIR, DATASET)
wf  = pd.read_csv(os.path.join(d, 'waveforms.csv')).to_numpy().astype(np.float32)
isi = pd.read_csv(os.path.join(d, 'isi_dist.csv')).to_numpy().astype(np.float32)
acg_path = os.path.join(d, 'acg.csv')
acg = (pd.read_csv(acg_path).to_numpy().astype(np.float32)
       if os.path.exists(acg_path) else np.zeros((len(wf), 100), dtype=np.float32))

# sanitize source NaNs/Infs (the loader also does this)
wf, isi, acg = (np.nan_to_num(a) for a in (wf, isi, acg))

labels_raw, keep = clean_labels(pd.read_csv(os.path.join(d, 'labels.csv')).iloc[:, -1])
wf, isi, acg, labels_raw = wf[keep], isi[keep], acg[keep], labels_raw[keep]

le = LabelEncoder()
labels = le.fit_transform(labels_raw)
num_classes = len(le.classes_)
print(f'{DATASET}: {len(labels)} labeled units, {num_classes} classes -> {list(le.classes_)}')

## 4. Build the dataset and model

`MultiModalEphysDataset` expects labels as `(class_idx, source_idx)` pairs; the source index
comes from `TECH`. When the config requests augmentations (the production default does), we
wrap it in `AugmentedMultiModalEphysDataset`, exactly as `scripts/train.py`.

In [ ]:
src_idx = TECHNOLOGY_IDS[TECH]
paired_labels = np.stack([labels, np.full(len(labels), src_idx, dtype=np.int64)], axis=1)

dataset = MultiModalEphysDataset(
    {'wave': wf, 'isi': isi, 'acg': acg}, labels=paired_labels, mode='multi', normalize=True)

config: CVAEConfig = getattr(ExperimentConfigs, CONFIG_NAME)()
config.beta = BETA
if config.use_augmentations and config.augment_pretraining:
    dataset = AugmentedMultiModalEphysDataset(dataset, config, phase='pretraining')

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                    collate_fn=none_safe_collate, drop_last=True)

model = MultiModalCVAE(
    modalities={'wave': 50, 'isi': 100, 'acg': 100}, z_dim=Z_DIM, config=config,
    num_sources=NUM_SOURCES, num_classes=num_classes)
train_module = MultiModalCVAETrainModule(base_model=model, config=config, learning_rate=1e-3)

## 5. Train

Single final checkpoint, no built-in validation/logging (those live in the benchmarking
repo). Increase `EPOCHS` and use a GPU for real training.

In [ ]:
os.makedirs(os.path.dirname(OUTPUT) or '.', exist_ok=True)
trainer = pl.Trainer(
    max_epochs=EPOCHS, accelerator=DEVICE, devices=1,
    enable_checkpointing=False, logger=False, log_every_n_steps=10)
trainer.fit(train_module, train_dataloaders=loader)
trainer.save_checkpoint(OUTPUT)
print(f'Saved checkpoint to {OUTPUT}')

## 6. Use the trained model

Reload the checkpoint through the public inference API and evaluate the embeddings with a
KNN probe (balanced accuracy), just like the inference notebook. `get_embeddings` expects
preprocessed inputs, so we resample + normalize here (same recipe the dataset used).

In [ ]:
import torch.nn.functional as F

def _resample_minmax(raw, target_len):
    t = torch.as_tensor(raw, dtype=torch.float32)
    if t.dim() == 1: t = t.unsqueeze(0)
    if t.shape[-1] != target_len:
        t = F.interpolate(t.unsqueeze(1), size=(target_len,), mode='linear',
                          align_corners=False).squeeze(1)
    mn, mx = t.amin(-1, keepdim=True), t.amax(-1, keepdim=True)
    return torch.nan_to_num((t - mn) / (mx - mn + 1e-8) * 2 - 1).numpy().astype(np.float32)

clf = HIPPIEClassifier.from_checkpoint(OUTPUT, device='cuda' if torch.cuda.is_available() else 'cpu')
emb = clf.get_embeddings(
    wave=_resample_minmax(wf, 50),
    isi=_resample_minmax(np.log(isi + 1.0), 100),
    acg=_resample_minmax(acg, 100),
    tech_id=TECH)

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
l2 = lambda x: x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

X_tr, X_te, y_tr, y_te = train_test_split(
    emb, labels_raw, test_size=0.2, stratify=labels_raw, random_state=SEED)
best_k, _ = select_k_via_cv(X_tr, y_tr)
knn = KNeighborsClassifier(n_neighbors=best_k, metric='cosine').fit(l2(X_tr), y_tr)
acc = balanced_accuracy_score(y_te, knn.predict(l2(X_te)))
print(f'balanced accuracy = {acc:.3f}  (chance = {1/num_classes:.3f})')

## Next steps

- **More data / epochs:** train on a larger corpus or raise `EPOCHS`; the released model was
  pretrained on the study's full collection of datasets.
- **Fine-tune instead of from-scratch:** start from the released weights
  (`HIPPIEClassifier.from_pretrained`) and continue training on your labels.
- **Command line:** the same run is `python scripts/train.py --dataset <name> --output <ckpt>`,
  and `hippie-cli embed --checkpoint <ckpt> ...` extracts embeddings.